# LLM 学習&推論体験セット

## 事前準備
1. （初回のみ）左メニュー **🔑 シークレット** に次を登録する。
   - **`HF_TOKEN`**（必須）… Hugging Face の Write トークン
   - **`HF_LORA_REPO`**（必須）… LoRA を push する先のモデル ID（例: `あなたのユーザー名/空のモデルリポジトリ名`）。Hugging Face Hub 上で **空のモデル用リポジトリ**を 1 つ作り、その ID を **`HF_LORA_REPO`** に入れておくこと。
2. **ランタイム → ランタイムのタイプを変更** で **GPU（T4 など）** を選ぶ。
3. **ランタイム → すべてのセルを実行**（または上から順に実行）。

## 1. 依存パッケージのインストール


In [ ]:
# Install Unsloth and other dependencies
!pip install -q "unsloth[colab-new]" peft accelerate bitsandbytes transformers trl huggingface_hub pyyaml datasets python-dotenv


## 2. GitHUBからソースコードをクローン

- `/content` 配下に git clone し、**`HF_TOKEN` / `HF_LORA_REPO`** をシークレットから環境変数に載せます。


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from typing import Optional


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_secret(name: str) -> Optional[str]:
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def _find_repo_root(start: Path) -> Optional[Path]:
    p = start.resolve()
    for _ in range(12):
        if (p / "training" / "finetune_script.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return None


REPO_OWNER = "TsutsujiStudyTeam"
REPO_NAME = "llm-model-generation"

if _in_colab():
    _go = (_colab_secret("GITHUB_REPO_OWNER") or "").strip()
    _gn = (_colab_secret("GITHUB_REPO_NAME") or "").strip()
    if _go:
        REPO_OWNER = _go
    if _gn:
        REPO_NAME = _gn
    content = Path("/content")
    content.mkdir(parents=True, exist_ok=True)
    repo_path = content / REPO_NAME
    if repo_path.exists():
        shutil.rmtree(repo_path, ignore_errors=True)
    os.chdir(content)
    github_token = _colab_secret("GITHUB_TOKEN")
    if github_token:
        clone_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    else:
        clone_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", clone_url, str(REPO_NAME)], check=True)
    os.chdir(repo_path)
else:
    root = _find_repo_root(Path.cwd())
    if root is None:
        raise RuntimeError(
            "リポジトリのルートを特定できませんでした。"
            "VS Code ではフォルダとして llm-model-generation を開き、ノートブックを開き直すか、"
            "カーネルのカレントディレクトリをリポジトリのルートにしてください。"
        )
    os.chdir(root)
    print("ローカル実行: git clone はスキップしました。")


def _load_dotenv_here() -> None:
    """カレントがリポジトリルートになったあとで .env を読む（ローカル用）。"""
    try:
        from dotenv import load_dotenv
    except ImportError:
        return
    env_path = Path.cwd() / ".env"
    if env_path.is_file():
        load_dotenv(env_path)
        print("（.env を読み込みました）")


_load_dotenv_here()

hf_token = (_colab_secret("HF_TOKEN") or os.environ.get("HF_TOKEN", "").strip())
if not hf_token and not _in_colab():
    import getpass

    print(
        "HF_TOKEN が未設定です。リポジトリのルートに .env を置き HF_TOKEN=... と記載するか（.env.example 参照）、"
        "このあと表示される入力に貼り付けてください（画面上には表示されません）。"
    )
    hf_token = getpass.getpass("HF_TOKEN: ").strip()
if not hf_token:
    raise RuntimeError(
        "HF_TOKEN がありません。\n"
        "・Google Colab: 左「🔑 シークレット」に名前 HF_TOKEN（値は Write トークン）を追加。\n"
        "・ローカル: リポジトリルートの .env、環境変数、または入力で渡す。"
    )
os.environ["HF_TOKEN"] = hf_token


def _colab_secret_into_env(name: str) -> None:
    if not _in_colab():
        return
    v = (_colab_secret(name) or "").strip()
    if v:
        os.environ[name] = v


_colab_secret_into_env("HF_MODEL_REPO")
_colab_secret_into_env("HF_LORA_REPO")

print("Working directory:", os.getcwd())


## 3. 学習の実行

In [ ]:
# subprocess + check=True だと CalledProcessError ばかり目立ち、本体の Traceback が分かりにくい。
# IPython でスクリプトを直接実行すると、エラーがこのセルにそのまま出る（コメントの下でも動く）。
from IPython import get_ipython

get_ipython().run_line_magic("run", "training/finetune_script.py")


## 4. 推論の実行

環境変数（セクション 2 のシークレット）の **`HF_LORA_REPO` があれば優先**し、なければ `params.yaml` の **hf_lora_repo** を使います。Unsloth 2026.x では **Hub の LoRA リポジトリ ID を `from_pretrained` に渡す**と、`adapter_config.json` が指す **ベースモデルを自動で読み込んでからアダプタを載せます**（このセルでは **`HF_MODEL_REPO` は参照しません**）。

> **古いノートブックの注意**: 下のコードが **`lora_adapter_merged` を `from_pretrained` で読む**実装なら **古い版**です。エラー例: `KeyError: 'shape'` / Mistral の tokenizer 警告。対処: [GitHub の `training/finetune.ipynb` を開き直す](https://github.com/TsutsujiStudyTeam/llm-model-generation/blob/main/training/finetune.ipynb)か、リポジトリを pull してから **セクション 4 のコードセル全体を置き換え**てください。

In [ ]:
# CPU のときは print のみ（raise しない）。このコメントが無い／raise があるなら古いセルです。
import yaml
from pathlib import Path

import torch

# Unsloth は import 時点で GPU を要求するため、先に CUDA の有無だけ判定する
if not torch.cuda.is_available():
    print(
        "[スキップ] CUDA がありません。簡易推論は GPU（例: Colab の T4）上でのみ実行されます。"
        "CPU の PC ではこのセルは何もしません（エラーではありません）。"
    )
else:
    import os
    from collections.abc import Mapping

    from unsloth import FastLanguageModel

    def _colab_userdata(name: str) -> str:
        try:
            from google.colab import userdata

            raw = userdata.get(name)
        except Exception:
            return ""
        return "" if raw is None else str(raw).strip()

    def _reject_merged_path(s: str, *, label: str) -> None:
        if "lora_adapter_merged" in s:
            raise RuntimeError(
                f"{label} に lora_adapter_merged が含まれています。"
                "マージ済みフォルダを単体で from_pretrained すると Transformers 5.x 付近で tokenizer エラー（例: KeyError 'shape'）になりがちです。"
                "Hub の LoRA モデル ID だけを from_pretrained に渡す実装に更新するか、GitHub の最新 finetune.ipynb を開き直してください。"
            )

    ROOT = Path.cwd()
    with open(ROOT / "training" / "params.yaml", encoding="utf-8") as f:
        params = yaml.safe_load(f)

    max_seq_length = int(params["max_seq_length"])
    lora_id = (
        os.environ.get("HF_LORA_REPO", "").strip()
        or _colab_userdata("HF_LORA_REPO")
        or str(params["hf_lora_repo"]).strip()
    )
    _reject_merged_path(lora_id, label="LoRA ID")
    if not lora_id or "YOUR_USERNAME" in lora_id:
        raise RuntimeError(
            "HF_LORA_REPO（シークレットまたは環境変数）か、params.yaml の hf_lora_repo を設定してください。"
        )

    _hf_tok = (os.environ.get("HF_TOKEN", "").strip() or _colab_userdata("HF_TOKEN")) or None
    # Unsloth 2026.x に load_lora_into_model は無い。PEFT リポジトリ ID を渡すとベース＋LoRA を解決して読む。
    _fp_kw = dict(
        model_name=lora_id,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    if _hf_tok:
        _fp_kw["token"] = _hf_tok
    model, tokenizer = FastLanguageModel.from_pretrained(**_fp_kw)
    FastLanguageModel.for_inference(model)

    messages = [
        {"role": "user", "content": "私は犬が好きです。これを否定文に変換してください。"},
    ]
    enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    device = model.device
    if isinstance(enc, torch.Tensor):
        input_ids = enc.to(device)
        attention_mask = None
    elif isinstance(enc, Mapping):
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
    else:
        input_ids = enc.to(device)
        attention_mask = None

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    gen_kw = dict(
        max_new_tokens=64,
        do_sample=False,
        # Unsloth 高速 KV + Transformers 5.3 付近で RoPE の cos と Q の長さがずれる例があるためキャッシュ無効化
        use_cache=False,
        pad_token_id=pad_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    with torch.inference_mode():
        if attention_mask is not None:
            outputs = model.generate(input_ids, attention_mask=attention_mask, **gen_kw)
        else:
            outputs = model.generate(input_ids, **gen_kw)

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
